In [1]:
import numpy as np
import sys
sys.path.append('./')
sys.path.append('../')
from cupiqp import SolverBase

%load_ext autoreload
%autoreload 2

In [2]:
P = np.array([[4.0, 1.0], [1.0, 2.0]])
c = np.array([1.0, 1.0])
A = np.array([[1.0, 1.0]])
b = np.array([1.0])
G = np.array([[1.0, 0.0], [0.0, 1.0], [-1.0, 0.0], [0.0, -1.0]])
h_u = np.array([0.5, 0.5, 1.0, 1.0])
h_l = np.array([-100., -100., -0.5, -0.5])
x_u = 100 * np.ones(2)
x_l = -100 * np.ones(2)

P = np.array([[6.0, 0.0], [0.0, 4.0]])
c = np.array([-1.0, -4.0])
A = np.array([[1.0, -2.0]])
b = np.array([1.0])
G = np.array([[1.0, -1.0], [2.0, 0.0]])
h_u = np.array([0.2, -1.0])
h_l = np.array([-10.0, -np.inf])
x_l = np.array([-1.0, -1.0])
x_u = np.array([1.0, 1.0])

# h_u = np.array([0.2, -1.0])
# h_l = np.array([-10.0, -np.inf])
x_l = np.array([-np.inf, -1.0])
x_u = np.array([1.0, np.inf])

# x_l = np.array([-1e10, -1.0])
# x_u = np.array([1.0, 1e10])


In [3]:
import cupy as cp

solver = SolverBase()
solver.settings.kkt_solver = 'dense_cholesky'
solver.settings.debug = False
solver.settings.verbose = True
solver.settings.max_iter = 20
with cp.cuda.Device(0):
	solver.setup(
		cp.array(P),
		cp.array(c), 
		cp.array(A), cp.array(b), cp.array(G), cp.array(h_u), cp.array(h_l), cp.array(x_u), cp.array(x_l))

	result = solver.solve()

print("Solution: ", solver._result.x)

Warp 1.11.0.dev20251115 initialized:
   Git commit: 62778e81d556f6573509dc314a49794370940d55
   CUDA Toolkit 12.8, Driver 13.0
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA GeForce RTX 5090" (31 GiB, sm_120, mempool enabled)
   Kernel cache:
     /home/fenglong/.cache/warp/1.11.0.dev20251115
Module cupiqp.dense.dense_kkt_solver 9a7cc38 load on device 'cuda:0' took 0.37 ms  (cached)
dense backend:
variables n = 2
equality constraints p = 1
inequality constraints m = 2
inequality lower bounds n_h_l = 1
inequality upper bounds n_h_u = 2
variable lower bounds n_x_l = 1
variable upper bounds n_x_u = 1

iter  prim_obj       dual_obj       duality_gap   prim_res      dual_res      rho         delta       mu          p_step   d_step
Module cupiqp.kkt_systems 0c0ae7f load on device 'cuda:0' took 0.30 ms  (cached)
  0    3.47993770e+00   -5.12159861e-02   3.53115369e+00   2.77451266e+00   5.90911583e+00   1.000000e-06   1.000000e-04   6.076640e+00   0.0000000   0.0000000
Solve

In [4]:
import cupy as cp
from cupy.sparse import csr_matrix

solver = SolverBase()
solver.settings.kkt_solver = 'sparse_ldlt'
# solver.settings.debug = True
solver.settings.verbose = True
solver.settings.max_iter = 10
with cp.cuda.Device(0):
	solver.setup(
		csr_matrix(cp.array(P)),
		cp.array(c), 
		csr_matrix(cp.array(A)), cp.array(b), csr_matrix(cp.array(G)), cp.array(h_u), cp.array(h_l), cp.array(x_u), cp.array(x_l))

	result = solver.solve()

print("Solution: ", solver._result.x)

No multithreading interface library was specified using the DirectSolverOptions. The performance of CPU operations like planning will be significantly lower than if you provide a multithreading library.


sparse backend:
variables n = 2, nnz(P) = 2
equality constraints p = 1, nnz(A) = 2
inequality constraints m = 2, nnz(G) = 3
inequality lower bounds n_h_l = 1
inequality upper bounds n_h_u = 2
variable lower bounds n_x_l = 1
variable upper bounds n_x_u = 1

iter  prim_obj       dual_obj       duality_gap   prim_res      dual_res      rho         delta       mu          p_step   d_step
  0    3.47993770e+00   -5.12159861e-02   3.53115369e+00   2.77451266e+00   5.90911583e+00   1.000000e-06   1.000000e-04   6.076640e+00   0.0000000   0.0000000
Solver::_calculate_step capturing CUDA graph (occurrence 1)...
  1    4.88333145e+00    4.17368144e+00   7.09650012e-01   1.57482096e-01   8.14332787e-02   6.483500e-08   6.483500e-06   3.939790e-01   0.9432492   0.9900000
  2    5.17040954e+00    5.86212378e+00   6.91714247e-01   6.97375177e-02   1.87574008e-01   2.474750e-08   4.643557e-07   2.821722e-02   0.5571393   0.9900000
  3    8.00688380e+00    3.61483876e+00   4.39204504e+00   7.69982560e